# 5.2 Regression

1. Load the Galton dataset into a Pandas DataFrame.
    * Dataset info: https://www.randomservices.org/random/data/Galton.html
    
2. Summarize the dataset:
    * Number of rows
    * Average height of male/female kids
    * Standard deviation of male/female kids
    
3. Create a training and test dataset. The test dataset should be at least 25%.

4. Create 2 regression models for predicting the child's height based on (i) father's height and (ii) mother's height.

5. Compute the model quality parameters: $R^{2}$ and $MSE$.

6. Create a multivariate regression model including both the mother's and father's height as features. How does $R^{2}$ change?


References:
* https://scikit-learn.org/stable/modules/linear_model.html
* https://scikit-learn.org/stable/model_selection.html
* <https://pygot.wordpress.com/2017/03/25/simple-linear-regression-with-galton/>


In [4]:
!pip install requests

In [5]:
%matplotlib inline
import csv
import requests # pip install requests for easy http request for CSV data
import numpy as np
import pandas as pd

In [6]:
df=pd.read_csv("https://ytliu0.github.io/Stat390EF-R-Independent-Study-archive/RMarkdownExercises/Galton.txt", sep="\t")

In [7]:
df.head()

,Family,Father,Mother,Gender,Height,Kids
0,1,78.5,67.0,M,73.2,4
1,1,78.5,67.0,F,69.2,4
2,1,78.5,67.0,F,69.0,4
3,1,78.5,67.0,F,69.0,4
4,2,75.5,66.5,M,73.5,4


In [44]:
# Task 5.2.2: Summarize the dataset
print("Summarize the dataset")
# Number of rows
print("Number of rows:", len(df))
# Average height
avg_H = df.groupby("Gender")["Height"].mean()
# Standard deviation of male/female kids
avg_Std = df.groupby("Gender")["Height"].std()


print("\nAverage height by gender:")
print(avg_H)
print("\nStandard deviation by gender:")
print(avg_Std)

Summarize the dataset
Number of rows: 898

Average height by gender:
Gender
F    64.110162
M    69.228817
Name: Height, dtype: float64

Standard deviation by gender:
Gender
F    2.370320
M    2.631594
Name: Height, dtype: float64


In [45]:
# Task 5.2.3  training and test dataset
from sklearn.model_selection import train_test_split
# Input features and target
X_father = df[['Father']]  # father's height as feature
X_mother = df[['Mother']]  # mother's height as feature
y        = df['Height']    # child's height as target

# Split into 75% training and 25% testing
X_father_train, X_father_test, y_father_train, y_father_test = train_test_split(
    X_father, y, test_size=0.25, random_state= 100)
X_mother_train, X_mother_test, y_mother_train, y_mother_test = train_test_split(
    X_mother, y, test_size=0.25, random_state = 100)

# Check sizes
print("Training set father size:", len(X_father_train))
print("Test set father size:", len(X_father_test))
print("Training set size Mother:", len(X_mother_train))
print("Test set size Mother:", len(X_mother_test))

Training set father size: 673
Test set father size: 225
Training set size Mother: 673
Test set size Mother: 225


In [46]:
# TASK 5.2.4: 2 regression models
from sklearn import linear_model


# Model 1: predict child height from father's height
print("TASK 5.2.4: 2 regression models")
model_father = linear_model.LinearRegression()
model_father.fit(X_father_train, y_father_train)
y_pred_father = model_father.predict(X_father_test)
print("print to check: Model 1 trained!")

# Model 2: predict child height from mother's height
model_mother = linear_model.LinearRegression()
model_mother.fit(X_mother_train, y_mother_train)
y_pred_mother = model_mother.predict(X_mother_test)
print("print to check Model 2 trained!")

TASK 5.2.4: 2 regression models
print to check: Model 1 trained!
print to check Model 2 trained!


In [47]:

# TASK 5.2.5: 𝑅2  and  𝑀𝑆𝐸
from sklearn.metrics import r2_score, mean_squared_error
print("TASK 5.2.5: R²  and  𝑀𝑆𝐸")
# Model 1
r2_father  = r2_score(y_father_test, y_pred_father)
mse_father = mean_squared_error(y_father_test, y_pred_father)

print("Model 1: Father's height")
print("R²  :", r2_father)
print("MSE :", mse_father)

# Model 2:
r2_mother  = r2_score(y_mother_test, y_pred_mother)
mse_mother = mean_squared_error(y_mother_test, y_pred_mother)

print("Model 2: Mother's height")
print("R²  :", r2_mother)
print("MSE :", mse_mother)
# Negative R^2 means the model is not useful


# R² values are low across both models, suggesting that a single parent's height
# is a poor predictor of the child's height on its own. Notably, the mother only
# model yields a negative R², meaning it performs worse than simply predicting
# the mean, a clear sign that one feature is insufficient. This motivates
# building a model that incorporates both parents' heights simultaneously.


TASK 5.2.5: R²  and  𝑀𝑆𝐸
Model 1: Father's height
R²  : 0.10481659057467108
MSE : 12.838265130120433
Model 2: Mother's height
R²  : 0.046241698199653514
MSE : 13.678316442913973


In [55]:
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
# TASK 5.2.6 : Multivariate regression

# Use BOTH father and mother height as features
print("multivariate regression")
X_FM = df[["Father", "Mother"]]
# Target: child's height
y      = df['Height']

# Split into training and test sets
X_FM_train, X_FM_test, y_FM_train, y_FM_test = train_test_split(
    X_FM, y, test_size=0.25, random_state= 100)
# Train the model
model_FM = linear_model.LinearRegression()
model_FM.fit(X_FM_train, y_FM_train)
y_pred_FM = model_FM.predict(X_FM_test)
# Compute R² and MSE
r2_FM  = r2_score(y_FM_test, y_pred_FM)
mse_FM = mean_squared_error(y_FM_test, y_pred_FM)

print("Model 3: parents")
print("R²  :", r2_FM)
print("MSE :", mse_FM)



# Adding both parents improves R² from 0.104 to 0.140
# But R² is still low overall because:
# 1. Gender is not included (influencing factor!)
# 2. Height depends on many other factors

multivariate regression
Model 3: parents
R²  : 0.14041420854108633
MSE : 12.327742199688824
